## 🔧 Environment Setup

Installs all required astronomy packages, extracts the latest exoplanet pipeline code from the attached Kaggle dataset, and copies models, target catalogs, and query caches to the active folders.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT SETUP
# Install libraries, load latest pipeline code, 
# create folder structure, and load cached resources.
# ═══════════════════════════════════════════════════════════
import os, sys, subprocess, shutil, zipfile
from datetime import date
from pathlib import Path

# ── User config ─────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/Iceandlava124/tess-exoplanet-pipeline.git"
KAGGLE_DATASET = "bhavishmehta/tess-exoplanet-discovery-results"
SESSION_LABEL  = date.today().strftime("%Y-%m-%d")
TIME_LIMIT_HRS = 8.5
DISK_LIMIT_GB  = 18
STARS_PER_RUN  = 800

WORKING_DIR = Path("/kaggle/working")
PIPELINE_DIR = WORKING_DIR / "pipeline"
RESULTS_DIR  = WORKING_DIR / "results"
INPUT_DIR   = Path("/kaggle/input/exoplanet-pipeline-resources")

# ── Install libraries ────────────────────────────────────────
print("Installing astronomy packages...")
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "lightkurve", "wotan", "batman-package", "astropy", "astroquery",
         "scipy", "scikit-learn", "imbalanced-learn", "tqdm", "joblib",
         "pandas", "numpy"],
        check=True, capture_output=True
    )
    print("SUCCESS: Packages installed.")
except Exception as e:
    print(f"Package install warning (may already be installed): {e}")

# ── Load pipeline source code ────────────────────────────────
# We prefer using the latest src.zip uploaded to the dataset,
# falling back to the GitHub repository.
try:
    if (INPUT_DIR / "src.zip").exists():
        print("Extracting source code from Kaggle dataset resources...")
        PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(INPUT_DIR / "src.zip", 'r') as zip_ref:
            zip_ref.extractall(PIPELINE_DIR)
        print("SUCCESS: Source code extracted from dataset.")
    elif (INPUT_DIR / "src").exists():
        print("Copying source code from Kaggle dataset resources...")
        shutil.copytree(INPUT_DIR / "src", PIPELINE_DIR / "src", dirs_exist_ok=True)
        print("SUCCESS: Source code copied from dataset.")
    else:
        print("Cloning latest pipeline from GitHub...")
        result = subprocess.run(
            ["git", "clone", "--depth=1", GITHUB_REPO, str(PIPELINE_DIR)],
            capture_output=True, text=True, timeout=180
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip())
        print("SUCCESS: Pipeline cloned from GitHub.")
except Exception as e:
    print(f"❌ Failed to load source code: {e}")

# ── Add pipeline to Python path ──────────────────────────────
for p in [str(PIPELINE_DIR), str(PIPELINE_DIR / "src"), str(WORKING_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Create output folder structure ──────────────────────────
for folder in ["KEEP", "FLAG", "DISCARD", "plots/flag_analysis", "reports", "figures", "data", "models"]:
    os.makedirs(RESULTS_DIR / folder, exist_ok=True)
    os.makedirs(WORKING_DIR / folder, exist_ok=True)

# ── Load models, targets, and cache database ──────────────────
# Copy resources from dataset to their active folders in the workspace
try:
    if INPUT_DIR.exists():
        print("Loading models, targets, and caching database from dataset...")
        
        # Copy models
        for m in ["random_forest.pkl", "cnn_classifier.h5"]:
            if (INPUT_DIR / m).exists():
                shutil.copy2(INPUT_DIR / m, WORKING_DIR / "models" / m)
                print(f"   Loaded model: {m}")
        
        # Copy targets
        if (INPUT_DIR / "training_targets.csv").exists():
            shutil.copy2(INPUT_DIR / "training_targets.csv", WORKING_DIR / "data" / "training_targets.csv")
            print("   Loaded targets catalog.")
            
        # Copy SQLite query cache
        if (INPUT_DIR / "pipeline_cache.db").exists():
            shutil.copy2(INPUT_DIR / "pipeline_cache.db", WORKING_DIR / "data" / "pipeline_cache.db")
            print("   Loaded SQLite query cache database.")
            
        print("SUCCESS: Dataset resources loaded.")
except Exception as e:
    print(f"Warning loading resources: {e}")

print(f"\n✅ Environment ready — Session: {SESSION_LABEL}")

## 📂 Resume From Last Week

Checks for previous discovery results and appends to them, avoiding redundant processing of already vetted stars.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — RESUME FROM LAST WEEK
# If a Kaggle dataset with previous results exists, copy it in
# so we never re-process stars we already analysed.
# ═══════════════════════════════════════════════════════════
import pandas as pd
from collections import Counter

INPUT_RESULTS_CSV = "/kaggle/input/tess-exoplanet-discovery-results/results.csv"
INPUT_CANDIDATES  = "/kaggle/input/tess-exoplanet-discovery-results/candidates_submission.csv"
OUTPUT_RESULTS_CSV = os.path.join(RESULTS_DIR, "results.csv")

already_done = set()
df_previous  = None

try:
    if os.path.exists(INPUT_RESULTS_CSV):
        shutil.copy2(INPUT_RESULTS_CSV, OUTPUT_RESULTS_CSV)
        df_previous = pd.read_csv(OUTPUT_RESULTS_CSV)
        if "tic_id" in df_previous.columns:
            already_done = set(df_previous["tic_id"].astype(int).tolist())
        print(f"📂 Loaded {len(df_previous)} previous results from Kaggle dataset.")
        print(f"   Already processed: {len(already_done)} unique stars")
        if "decision" in df_previous.columns:
            counts = Counter(df_previous["decision"].fillna("UNKNOWN"))
            print(f"   KEEP:    {counts.get('KEEP', 0)}")
            print(f"   FLAG:    {counts.get('FLAG', 0)}")
            print(f"   DISCARD: {counts.get('DISCARD', 0)}")
    else:
        print("🆕 Fresh start — no previous results found")
except Exception as e:
    print(f"❌ Error loading previous results: {e}")

try:
    if os.path.exists(INPUT_CANDIDATES):
        shutil.copy2(INPUT_CANDIDATES, os.path.join(RESULTS_DIR, "candidates_submission.csv"))
        print("   Candidates file copied.")
except Exception as e:
    print(f"   Warning: could not copy candidates file: {e}")

print(f"✅ Resume check complete — {len(already_done)} stars will be skipped.")

## 🎯 This Week's Targets

Builds a prioritised target list of candidate stars from the TESS Input Catalog (TIC) applying physical filters.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — BUILD TARGET LIST
# Query the TESS Input Catalog to find this week's candidate
# stars. We apply physical filters to focus on stars where
# TESS is likely to detect a planet, and skip stars we already
# processed in previous weeks.
# ═══════════════════════════════════════════════════════════
import sys, os
# Make sure pipeline directory and working directory are in path
for path_dir in ["/kaggle/working/pipeline", "/kaggle/working"]:
    if path_dir not in sys.path:
        sys.path.insert(0, path_dir)

try:
    from kaggle_discovery_runner import build_target_list
except Exception as e_imp:
    print(f"❌ Failed to import build_target_list: {e_imp}")
    print("Python Path is:", sys.path)
    raise e_imp

TARGETS_CSV = os.path.join(RESULTS_DIR if 'RESULTS_DIR' in globals() else '/kaggle/working/results', "this_week_targets.csv")

try:
    targets = build_target_list(
        n_targets            = STARS_PER_RUN,
        already_processed    = already_done if 'already_done' in globals() else set(),
        mag_range            = (8, 13),
        teff_range           = (3500, 7000),
        radius_range         = (0.5, 2.0),
        exclude_giants       = True,
        exclude_known_contaminated = True,
        prioritise_multi_sector    = True,
        prioritise_not_in_toi      = True,
    )
    targets.to_csv(TARGETS_CSV, index=False)
    n = len(targets)
    est_hrs = n * 45 / 3600
    print(f"🎯 {n} stars queued for this session")
    print(f"   Estimated time: {est_hrs:.1f} hours at ~45s per star")
    print(f"   Target list saved to: {TARGETS_CSV}")
except Exception as e:
    print(f"❌ Target list build failed: {e}")
    import traceback; traceback.print_exc()
    targets = pd.DataFrame(columns=["tic_id"])

print(f"✅ Target list ready — {len(targets)} stars")

## 🔭 Autonomous Discovery — ~8.5 hrs

Executes the exoplanet detection pipeline v2.0 on the selected targets.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — AUTONOMOUS DISCOVERY
# This runs the full pipeline on every star in the target list.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import run_discovery_session
except ImportError:
    sys.path.insert(0, str(PIPELINE_DIR))
    from kaggle_discovery_runner import run_discovery_session

session_summary = {}
if len(targets) == 0:
    print("❌ No targets to process — re-run Cell 3 first.")
else:
    try:
        session_summary = run_discovery_session(
            targets           = targets,
            output_dir        = RESULTS_DIR,
            models_dir        = os.path.join(WORKING_DIR, "models"),
            time_limit_hours  = TIME_LIMIT_HRS,
            disk_limit_gb     = DISK_LIMIT_GB,
            save_every_n      = 50,
            session_label     = SESSION_LABEL,
            # Pipeline feature flags
            run_flag_analyzer      = True,
            run_candidate_export   = True,
            run_toi_crosscheck     = True,
            alias_rejection        = True,
            # Physical plausibility filters
            max_planet_radius_earth = 25.0,
            min_transit_snr         = 5.0,
            min_depth_ppm           = 100,
            # Log file
            log_file = os.path.join(RESULTS_DIR, "discovery_log.txt"),
        )
        n_proc    = session_summary.get("stars_processed", 0)
        n_skip    = session_summary.get("stars_skipped", 0)
        elapsed   = session_summary.get("elapsed_hours", 0)
        print(f"✅ Discovery session complete")
        print(f"   Stars processed: {n_proc}")
        print(f"   Stars skipped:   {n_skip}")
        print(f"   Time elapsed:    {elapsed:.1f} hrs")
    except Exception as e:
        print(f"❌ Discovery session crashed: {e}")
        import traceback; traceback.print_exc()
        session_summary = {}


## 📊 Session Summary

Generates statistics and diagnostic tables highlighting potential new exoplanet discoveries.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — SESSION SUMMARY
# Print final statistics and highlight new planet discoveries.
# ═══════════════════════════════════════════════════════════
try:
    from kaggle_discovery_runner import generate_session_summary
except ImportError:
    sys.path.insert(0, str(PIPELINE_DIR))
    from kaggle_discovery_runner import generate_session_summary

try:
    summary = generate_session_summary(
        results_dir   = RESULTS_DIR,
        session_label = SESSION_LABEL,
        session_data  = session_summary,
    )

    n_attempted  = summary.get("stars_attempted", 0)
    n_processed  = summary.get("stars_processed", 0)
    n_skipped    = summary.get("stars_skipped", 0)
    n_alias      = summary.get("alias_discards", 0)
    elapsed      = summary.get("elapsed_hours", 0)
    throughput   = n_processed / elapsed if elapsed > 0 else 0
    n_keep       = summary.get("keep", 0)
    n_flag       = summary.get("flag", 0)
    n_discard    = summary.get("discard", 0)
    n_new        = summary.get("new_discoveries", 0)
    n_cumulative = summary.get("cumulative_total", 0)

    print("=" * 50)
    print(f"  TESS DISCOVERY RUN -- {SESSION_LABEL}")
    print("=" * 50)
    print(f"  Stars attempted:   {n_attempted}")
    print(f"  Stars processed:   {n_processed}")
    print(f"  Skipped (no data): {n_skipped}")
    print(f"  Alias discards:    {n_alias}")
    print(f"  Time elapsed:      {elapsed:.1f} hrs")
    print(f"  Throughput:        {throughput:.0f} stars/hr")
    print("-" * 50)
    print(f"  KEEP:    {n_keep} ({n_new} potential new discoveries)")
    print(f"  FLAG:    {n_flag} (human review needed)")
    print(f"  DISCARD: {n_discard}")
    print("-" * 50)
    print(f"  Cumulative total:  {n_cumulative} stars all-time")
    print("=" * 50)
except Exception as e:
    print(f"❌ Summary generation failed: {e}")
    summary = {}

try:
    new_disc_file = os.path.join(RESULTS_DIR, "new_discoveries.txt")
    if os.path.exists(new_disc_file) and os.path.getsize(new_disc_file) > 0:
        with open(new_disc_file, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        if lines:
            print("\nPOTENTIAL NEW DISCOVERIES NOT IN TOI CATALOG:")
            for line in lines:
                print(f"  {line}")
except Exception as e:
    print(f"   Warning: could not read new_discoveries.txt: {e}")

try:
    log_path = os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md")
    write_header = not os.path.exists(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        if write_header:
            f.write("# TESS Discovery Log\n\n")
            f.write("| Date | Processed | KEEP | FLAG | DISCARD | New | Notes |\n")
            f.write("|------|-----------|------|------|---------|-----|-------|\n")
        notes = f"Alias discards: {summary.get('alias_discards', 0)}"
        f.write(f"| {SESSION_LABEL} | {n_processed} | {summary.get('keep',0)} | "
                f"{summary.get('flag',0)} | {summary.get('discard',0)} | "
                f"{summary.get('new_discoveries',0)} | {notes} |\n")
    print("✅ Discovery log updated")
except Exception as e:
    print(f"❌ Log update failed: {e}")

## 💾 Save & Export

Saves the cumulative CSV and pushes the updated results database back to Kaggle.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — SAVE & EXPORT
# ═══════════════════════════════════════════════════════════
import shutil

export_map = {
    os.path.join(RESULTS_DIR, "results.csv"):                   "results_cumulative.csv",
    os.path.join(RESULTS_DIR, "candidates_submission.csv"):     "candidates_all.csv",
    os.path.join(RESULTS_DIR, "manual_review_queue.csv"):       "review_queue_latest.csv",
    os.path.join(RESULTS_DIR, "DISCOVERY_LOG.md"):              "DISCOVERY_LOG.md",
    os.path.join(RESULTS_DIR, "new_discoveries.txt"):           "new_discoveries.txt",
}

for src, dest_name in export_map.items():
    try:
        if os.path.exists(src) and os.path.getsize(src) > 0:
            dest = os.path.join(WORKING_DIR, dest_name)
            shutil.copy2(src, dest)
            print(f"Saved: {dest_name}")
        else:
            print(f"   Skipped (empty or missing): {dest_name}")
    except Exception as e:
        print(f"   Failed to copy {dest_name}: {e}")

try:
    n_processed_final = session_summary.get("stars_processed", 0)
    commit_msg = f"Auto-update: {SESSION_LABEL} -- {n_processed_final} stars processed"
    print(f"Pushing results to Kaggle dataset '{KAGGLE_DATASET}'...")
    result = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", RESULTS_DIR,
         "-m", commit_msg,
         "--dir-mode", "zip"],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode == 0:
        print("Dataset updated successfully.")
    else:
        print(f"   Kaggle push warning: {result.stderr.strip()}")
except Exception as e:
    print(f"   Kaggle push failed: {e}")

print("\nAll done!")